## What this does

A **smart money assistant** built with the **OpenAI Agents SDK** and a **custom MCP server** (`expense_bank_server.py`):

- **Track expenses** — SQLite-backed `add_expense`, `list_expenses`, `summarize_by_category`.
- **Bank PDFs** — put copy-only statement PDFs in `bank_pdfs/`, then the agent uses `list_bank_pdfs` and `extract_bank_pdf_text` to pull text and map lines to categories (the model parses messy text; image-only PDFs need OCR outside this demo).
- **Insights** — `compute_insights` returns totals, flags when food-related spend is a high share of the period, and a **heuristic savings line** (e.g. “~20% off food/dining could free about $X for the same window”).


In [1]:
import os
import sys
from pathlib import Path

_here = Path.cwd()
_six_mcp = None
for d in [_here, *_here.parents]:
    if (d / "accounts_server.py").is_file():
        _six_mcp = d
        break
if _six_mcp is None:
    raise RuntimeError("Open this notebook from the agents repo (or cd into 6_mcp).")

os.chdir(_six_mcp)
_contrib = _six_mcp / "community_contributions" / "temmyogunbo"
(_contrib / "bank_pdfs").mkdir(parents=True, exist_ok=True)

from dotenv import load_dotenv
from IPython.display import Markdown, display

from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio

load_dotenv(_six_mcp.parent / ".env", override=True)
load_dotenv(override=True)

True

In [2]:
expense_mcp_params = {
    "command": "uv",
    "args": ["run", "community_contributions/temmyogunbo/expense_bank_server.py"],
}

mcp_servers = [MCPServerStdio(expense_mcp_params, client_session_timeout_seconds=120)]

In [3]:
USER_PROMPT = """
You are helping me with personal budgeting for March 2026.
1) If there are PDFs in bank_pdfs, list them and extract text from the first one; parse obvious debits into add_expense calls
   (use categories from the expense://categories resource when helpful).
2) For 2026-03-01 through 2026-03-31, call summarize_by_category and compute_insights.
3) Reply in clear bullets: totals, any “overspending on food” style warnings, and the savings ideas with dollar amounts from compute_insights.
Be explicit if the PDF had no parseable lines.
"""

instructions = """
You are a supportive personal finance coach. You only use tools from the expense MCP server.
Never invent dollar amounts: derive them from list_expenses, summarize_by_category, compute_insights, or from PDF text you extracted.
When parsing PDFs, merchants and dates may be messy — make reasonable guesses and note uncertainty.
Keep advice general (not investment or tax advice).
"""

agent = Agent(
    name="ExpenseCoach",
    instructions=instructions,
    model="gpt-4.1-mini",
    mcp_servers=mcp_servers,
)

In [ ]:
for server in mcp_servers:
    await server.connect()

with trace("ExpenseInsights"):
    result = await Runner.run(agent, USER_PROMPT.strip(), max_turns=35)

display(Markdown(result.final_output))